# 00 - Data Audit

Audit the lost-language corpus for basic structural properties:
- Row counts and missing values
- Token coverage per document/line
- Vocabulary size and type-token ratio
- Boundary integrity (no cross-line contamination)

**Data source:** `data/processed/lost_tokens.csv` (or synthetic benchmark).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/processed/lost_tokens.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Missing per column:\n{df.isnull().sum()}")

In [ ]:
# Basic coverage
print(f"Documents: {df['doc_id'].nunique()}")
print(f"Lines: {df.groupby(['doc_id', 'line_id']).ngroups}")
print(f"Tokens: {len(df)}")
print(f"Vocabulary: {df['token'].nunique()}")
print(f"Type-Token Ratio: {df['token'].nunique() / len(df):.4f}")

In [ ]:
# Line length distribution
line_lengths = df.groupby(['doc_id', 'line_id']).size()
print(line_lengths.describe())

plt.figure(figsize=(8, 4))
plt.hist(line_lengths, bins=range(1, line_lengths.max()+2), edgecolor='black')
plt.xlabel("Tokens per line")
plt.ylabel("Frequency")
plt.title("Line length distribution")
plt.savefig("../reports/figures/line_length_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Boundary check: ensure line_id resets per doc_id
boundary_check = df.groupby('doc_id')['line_id'].min().reset_index()
boundary_check.columns = ['doc_id', 'min_line_id']
print("Min line_id per doc (should be 1):")
print(boundary_check)

assert boundary_check['min_line_id'].min() == 1, "Line IDs do not reset per document!"
print("\n[OK] Boundary integrity verified.")